# FastAPI Shopping Cart System - Phase 1 Testing
## Customer 1 and Customer 2 Flows

This notebook tests the complete shopping cart flow for two customers with automatic verification of cart totals, orders, and state management.

In [ ]:
import requests
import json
import time

# Configuration
BASE_URL = "http://127.0.0.1:8000"

def print_section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}\n")

def print_response(label, response):
    print(f"{label}")
    print(f"Status Code: {response.status_code}")
    print(f"Response:\n{json.dumps(response.json(), indent=2)}\n")

# Check if server is running
try:
    response = requests.get(f"{BASE_URL}/cart")
    print("✓ Server is running and accessible!")
except:
    print("✗ Server is not running. Please restart the server with:")
    print("  uvicorn main:app --reload")


## STEP 1: Setup and Server Restart

**Before running this test:**
1. Open a terminal and press Ctrl+C to stop any running server
2. Run: `uvicorn main:app --reload` 
3. Ensure a fresh, empty cart state

This notebook will execute the complete Customer 1 and Customer 2 flows in sequence.

## STEP 2️⃣: Customer 1 - Add Items to Cart

**Action:** Add Wireless Mouse (qty 1) and Pen Set (qty 3) to the cart

Expected Results:
- Both items should be added successfully (Status 200)
- Cart should contain 2 items

In [ ]:
print_section("CUSTOMER 1 - PHASE 1")

# Add Wireless Mouse (product_id=1, quantity=1)
print("Adding Wireless Mouse (qty 1)...")
response1 = requests.post(f"{BASE_URL}/cart/add?product_id=1&quantity=1")
print_response("Wireless Mouse Added:", response1)
assert response1.status_code == 200, "Failed to add Wireless Mouse"

# Add Pen Set (product_id=4, quantity=3)
print("Adding Pen Set (qty 3)...")
response2 = requests.post(f"{BASE_URL}/cart/add?product_id=4&quantity=3")
print_response("Pen Set Added:", response2)
assert response2.status_code == 200, "Failed to add Pen Set"

print("✓ Both items added to cart successfully!")

## STEP 3️⃣: Customer 1 - Verify Cart Total

**Verification:** GET /cart should show grand_total = ₹646

Calculation:
- Wireless Mouse: ₹499 × 1 = ₹499
- Pen Set: ₹49 × 3 = ₹147
- **Total: ₹646**

In [ ]:
print("\nVerifying cart total...")
response_cart = requests.get(f"{BASE_URL}/cart")
print_response("Current Cart:", response_cart)

cart_data = response_cart.json()
grand_total = cart_data.get("grand_total", 0)
expected_total = 646

print(f"Expected Total: ₹{expected_total}")
print(f"Actual Total: ₹{grand_total}")

if grand_total == expected_total:
    print(f"✓ Cart total is CORRECT! ₹{grand_total}")
else:
    print(f"✗ Cart total is INCORRECT! Expected ₹{expected_total}, got ₹{grand_total}")

assert grand_total == expected_total, f"Cart total mismatch: {grand_total} != {expected_total}"

## STEP 4️⃣: Customer 1 - Checkout

**Action:** Checkout as Customer 1 with delivery address (minimum 10 characters)

Expected Results:
- Checkout succeeds (Status 200)
- 2 orders are created (one for each product)
- Cart is cleared after checkout

In [ ]:
print("\nChecking out as Customer 1...")

checkout_data_c1 = {
    "customer_name": "Customer 1",
    "delivery_address": "123 MG Road, Bangalore 560001"
}

response_checkout_c1 = requests.post(
    f"{BASE_URL}/cart/checkout",
    json=checkout_data_c1
)
print_response("Customer 1 Checkout Response:", response_checkout_c1)
assert response_checkout_c1.status_code == 200, "Checkout failed for Customer 1"

checkout_result = response_checkout_c1.json()
orders_placed = checkout_result.get("orders_placed", 0)
checkout_total = checkout_result.get("grand_total", 0)

print(f"Orders Placed: {orders_placed}")
print(f"Checkout Total: ₹{checkout_total}")

assert orders_placed == 2, f"Expected 2 orders, got {orders_placed}"
assert checkout_total == 646, f"Expected total ₹646, got ₹{checkout_total}"

print("✓ Customer 1 checkout successful with 2 orders!")

## STEP 5️⃣: Customer 1 - Verify Orders

**Verification:**
- Cart should be empty (no items)
- GET /orders should show 2 orders (one for Wireless Mouse, one for Pen Set)

In [ ]:
print("\nVerifying cart is empty after checkout...")
response_empty_cart = requests.get(f"{BASE_URL}/cart")
empty_cart = response_empty_cart.json()
print_response("Cart After Checkout:", response_empty_cart)

assert empty_cart.get("item_count", 0) == 0, "Cart should be empty after checkout"
print("✓ Cart is empty!")

print("\nFetching all orders...")
response_orders = requests.get(f"{BASE_URL}/orders")
print_response("All Orders:", response_orders)

orders_data = response_orders.json()
all_orders = orders_data.get("orders", [])
total_orders = orders_data.get("total_orders", 0)

print(f"Total Orders in System: {total_orders}")
print(f"Customer 1 Orders:")
for order in all_orders:
    if order.get("customer_name") == "Customer 1":
        print(f"  - Order ID: {order['order_id']}, Product: {order['product']}, Qty: {order['quantity']}, Price: ₹{order['total_price']}")

assert total_orders == 2, f"Expected 2 orders after Customer 1 checkout, got {total_orders}"
print("✓ Customer 1 verification complete! 2 orders placed.")

---
## CUSTOMER 2 - Phase 2

**Note:** No server restart! Using the same session as Customer 1.

## STEP 6️⃣: Customer 2 - Add Items to Cart

**Action:** Add Notebook (qty 2) and Wireless Mouse (qty 1) to cart

Expected Results:
- Both items added successfully
- Cart should now contain 2 items (no reset between customers)

In [ ]:
print_section("CUSTOMER 2 - PHASE 2 (Same Session)")

# Add Notebook (product_id=2, quantity=2)
print("Adding Notebook (qty 2)...")
response3 = requests.post(f"{BASE_URL}/cart/add?product_id=2&quantity=2")
print_response("Notebook Added:", response3)
assert response3.status_code == 200, "Failed to add Notebook"

# Add Wireless Mouse (product_id=1, quantity=1)
print("Adding Wireless Mouse (qty 1)...")
response4 = requests.post(f"{BASE_URL}/cart/add?product_id=1&quantity=1")
print_response("Wireless Mouse Added:", response4)
assert response4.status_code == 200, "Failed to add Wireless Mouse"

print("✓ Both items added to cart for Customer 2!")

## STEP 7️⃣: Customer 2 - Remove Notebook from Cart

**Action:** DELETE /cart/2 to remove Notebook

Expected Results:
- Notebook (product_id=2) removed successfully
- Only Wireless Mouse remains in cart

In [ ]:
print("\nRemoving Notebook (product_id=2) from cart...")
response_delete = requests.delete(f"{BASE_URL}/cart/2")
print_response("Notebook Removed:", response_delete)
assert response_delete.status_code == 200, "Failed to remove Notebook"

# Verify only Wireless Mouse remains
print("\nVerifying cart after removal...")
response_cart_after_delete = requests.get(f"{BASE_URL}/cart")
cart_after_delete = response_cart_after_delete.json()
print_response("Cart After Removal:", response_cart_after_delete)

item_count = cart_after_delete.get("item_count", 0)
assert item_count == 1, f"Expected 1 item after deletion, got {item_count}"

grand_total_after_delete = cart_after_delete.get("grand_total", 0)
expected_mouse_price = 499

print(f"Items in Cart: {item_count}")
print(f"Grand Total: ₹{grand_total_after_delete}")
assert grand_total_after_delete == expected_mouse_price, f"Expected ₹{expected_mouse_price}, got ₹{grand_total_after_delete}"

print("✓ Notebook removed! Only Wireless Mouse (₹499) remains in cart.")

## STEP 8️⃣: Customer 2 - Checkout

**Action:** Checkout as Customer 2 with only Wireless Mouse in cart

Expected Results:
- Checkout succeeds with 1 order (only Wireless Mouse)
- Order total: ₹499
- Cart is cleared

In [ ]:
print("\nChecking out as Customer 2...")

checkout_data_c2 = {
    "customer_name": "Customer 2",
    "delivery_address": "456 Raj Path, Mumbai 400001"
}

response_checkout_c2 = requests.post(
    f"{BASE_URL}/cart/checkout",
    json=checkout_data_c2
)
print_response("Customer 2 Checkout Response:", response_checkout_c2)
assert response_checkout_c2.status_code == 200, "Checkout failed for Customer 2"

checkout_result_c2 = response_checkout_c2.json()
orders_placed_c2 = checkout_result_c2.get("orders_placed", 0)
checkout_total_c2 = checkout_result_c2.get("grand_total", 0)

print(f"Orders Placed: {orders_placed_c2}")
print(f"Checkout Total: ₹{checkout_total_c2}")

assert orders_placed_c2 == 1, f"Expected 1 order for Customer 2, got {orders_placed_c2}"
assert checkout_total_c2 == 499, f"Expected total ₹499, got ₹{checkout_total_c2}"

print("✓ Customer 2 checkout successful with 1 order!")

## STEP 9️⃣: Customer 2 - Verify Final Orders

**Verification:** GET /orders should now show **3 total orders**
- 2 orders from Customer 1 (Wireless Mouse + Pen Set)
- 1 order from Customer 2 (Wireless Mouse)

**Summary of Expected Orders:**
1. Order 1: Customer 1 - Wireless Mouse (₹499)
2. Order 2: Customer 1 - Pen Set (₹147)
3. Order 3: Customer 2 - Wireless Mouse (₹499)

In [ ]:
print("\nFetching final orders summary...")
response_final_orders = requests.get(f"{BASE_URL}/orders")
print_response("Final Orders:", response_final_orders)

final_orders_data = response_final_orders.json()
all_final_orders = final_orders_data.get("orders", [])
total_final_orders = final_orders_data.get("total_orders", 0)

print(f"\n{'='*60}")
print(f"  FINAL ORDERS SUMMARY")
print(f"{'='*60}")
print(f"Total Orders in System: {total_final_orders}\n")

# Group orders by customer
customers_orders = {}
for order in all_final_orders:
    customer = order.get("customer_name")
    if customer not in customers_orders:
        customers_orders[customer] = []
    customers_orders[customer].append(order)

# Display orders by customer
for idx, (customer, orders_list) in enumerate(customers_orders.items(), 1):
    print(f"{idx}. {customer}:")
    for order in orders_list:
        print(f"   - Order #{order['order_id']}: {order['product']} (Qty: {order['quantity']}) = ₹{order['total_price']}")
    customer_total = sum(o['total_price'] for o in orders_list)
    print(f"   Subtotal: ₹{customer_total}\n")

# Final assertions
assert total_final_orders == 3, f"Expected 3 total orders, got {total_final_orders}"

c1_orders = len(customers_orders.get("Customer 1", []))
c2_orders = len(customers_orders.get("Customer 2", []))

assert c1_orders == 2, f"Expected 2 orders for Customer 1, got {c1_orders}"
assert c2_orders == 1, f"Expected 1 order for Customer 2, got {c2_orders}"

print(f"{'='*60}")
print("✓ ALL TESTS PASSED!")
print(f"  Customer 1: {c1_orders} orders")
print(f"  Customer 2: {c2_orders} orders")
print(f"  Total: {total_final_orders} orders")
print(f"{'='*60}")

# Display overall totals
total_revenue = sum(o['total_price'] for o in all_final_orders)
print(f"\nTotal System Revenue: ₹{total_revenue}")


---

## 🎉 Test Execution Complete

### Summary
This notebook successfully tested:

✅ **Customer 1 Flow:**
- Added 2 products (Mouse + Pen Set)
- Verified cart total: ₹646
- Created 2 orders after checkout
- Cart cleared post-checkout

✅ **Customer 2 Flow (Same Session):**
- Added 2 products (Notebook + Mouse)
- Removed 1 product (Notebook)
- Checked out with only Mouse
- Created 1 order

✅ **Final Verification:**
- Total 3 orders in system (2 from C1, 1 from C2)
- System revenue: ₹1,145 (₹646 + ₹499)

### Key Test Results
| Metric | Value |
|--------|-------|
| Total Orders | 3 ✓ |
| Customer 1 Orders | 2 ✓ |
| Customer 2 Orders | 1 ✓ |
| System Revenue | ₹1,145 |
| Cart State After Checkout | Empty ✓ |

**Status: ALL TESTS PASSED! 🚀**